# Calibrating several models together

Sometimes one calibration involves more than one model: several cities that share most of their structure, or a transmission model and a few natural-history models of different kinds. They may share a parameter, or have parameters that are related but not equal, such as recovery rates that vary around a common average (a *hierarchy*).

Each `BayesianModel` names its parameters locally, so both models below have a parameter called `recovery`. Numpyro site names are global, and the two `recovery` values must be allowed to differ. This page shows the helpers that handle the naming:

1. two models that use the same parameter name;
2. checking names before scoring;
3. a joint model that gives each model its own prefix and a hierarchical `recovery`;
4. fitting it and replaying each model from the joint posterior.

The page runs in a minute or two.

## Two models with the same parameter name

The first is a city SIR with an infection rate and a recovery rate, observed as the number of infectious people every 2 days for 20 days. The second is a cohort of 300 infectious people who recover at rate `recovery`, observed daily for 14 days. Both use the default adaptive solver, with `dt` as the first step.

In [ ]:
import warnings

import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
import optax
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from numpyro.infer import MCMC, NUTS
from plotly.subplots import make_subplots

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import (
    BayesianModel,
    NegativeBinomial,
    NumpyroPosterior,
    Poisson,
    Uniform,
)
from summer4.epi.calibration import workflow as wf

# The city: S -> I -> R with frequency-dependent infection.
state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
city_map = PropertyMap.from_property(state).stratify(pop)
city_flows = FlowModel(city_map)
city_flows.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]])),
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("infection"),
        ),
    )
)
city_flows.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
city_cm = city_flows.compile()

# The cohort: I -> R only.
cohort_state = Property("state", ("I", "R"))
cohort_map = PropertyMap.from_property(cohort_state)
cohort_flows = FlowModel(cohort_map)
cohort_flows.add_flow(
    TransitionFlow("recovery", cohort_state["I"], cohort_state["R"], Param("recovery"))
)
cohort_cm = cohort_flows.compile()

## Synthetic data

We pick true values, run each model once, and add noise with a fixed seed: negative binomial scatter for the city counts and Poisson scatter for the cohort counts. The true recovery rates differ a little between the two, as they would in a hierarchy.

In [ ]:
true = {"infection": 0.55, "city.recovery": 0.28, "cohort.recovery": 0.40}
city_times = np.arange(0.0, 21.0, 2.0)
cohort_times = np.arange(0.0, 15.0, 1.0)
rng = np.random.default_rng(11)

# City.
city_y0_values = np.zeros(city_map.size)
city_y0_values[city_map.select(state["S"])] = 1e6 - 20.0
city_y0_values[city_map.select(state["I"])] = 20.0
city_y0 = PropertyData.wrap(city_map, city_y0_values)
city_I = Compartments(where=state["I"])
city_kwargs = {"t0": 0.0, "t1": 20.0, "dt": 0.25}
clean = city_cm.run(
    {"infection": true["infection"], "recovery": true["city.recovery"]},
    city_y0,
    save=SavePlan(requests={"I": SaveRequest(city_I, ts=city_times)}),
    **city_kwargs,
)
expected = np.asarray(clean["I"].values.data).reshape(-1)
city_obs = rng.negative_binomial(20.0, 20.0 / (20.0 + expected)).astype(float)

# Cohort.
cohort_y0_values = np.zeros(cohort_map.size)
cohort_y0_values[cohort_map.select(cohort_state["I"])] = 300.0
cohort_y0 = PropertyData.wrap(cohort_map, cohort_y0_values)
cohort_I = Compartments(where=cohort_state["I"])
cohort_kwargs = {"t0": 0.0, "t1": 14.0, "dt": 0.25}
clean = cohort_cm.run(
    {"recovery": true["cohort.recovery"]},
    cohort_y0,
    save=SavePlan(requests={"I": SaveRequest(cohort_I, ts=cohort_times)}),
    **cohort_kwargs,
)
expected = np.asarray(clean["I"].values.data).reshape(-1)
cohort_obs = rng.poisson(expected).astype(float)

## One `BayesianModel` each

Each model gets its own priors and its own target. The city carries a `fixed_params` entry, a development default for `recovery` of 0.25, which is used whenever `recovery` is not supplied. `prior_names()` lists the parameters each model samples: both have `recovery`.

In [ ]:
city = BayesianModel(
    city_cm,
    {"recovery": 0.25},
    priors=(Uniform("infection", 0.1, 1.5), Uniform("recovery", 0.02, 1.0)),
    targets=TargetSet(
        targets=(
            Target(
                key="I",
                times=city_times,
                values=city_obs,
                quantity=city_I,
                likelihood=NegativeBinomial(dispersion=20.0),
            ),
        )
    ),
    y0=city_y0,
    run_kwargs=city_kwargs,
)
cohort = BayesianModel(
    cohort_cm,
    {},
    priors=(Uniform("recovery", 0.02, 1.0),),
    targets=TargetSet(
        targets=(
            Target(
                key="I",
                times=cohort_times,
                values=cohort_obs,
                quantity=cohort_I,
                likelihood=Poisson(),
            ),
        )
    ),
    y0=cohort_y0,
    run_kwargs=cohort_kwargs,
)

pd.Series({"city": city.prior_names(), "cohort": cohort.prior_names()}, name="prior_names()")

## Check names before you score

`log_likelihood` takes a dict of parameter values. It fills any name you leave out from `fixed_params`, and it ignores any name it does not know. So a misspelled name does not fail: the model is scored with the fixed value instead, and the number looks fine.

`check_params` compares the names in a dict with the model's prior sites and fixed names, and raises `ValueError` if a prior site is missing or a name is unknown. `log_likelihood(..., strict=True)` runs that check first. Use `strict=True` whenever you build the dict by hand. `factor` (below) always does.

The cell scores the city at its true values, then with `recovery` misspelled, which silently uses 0.25, and then asks `check_params` about the misspelled dict.

In [ ]:
good = {"infection": true["infection"], "recovery": true["city.recovery"]}
typo = {"infection": true["infection"], "recovry": true["city.recovery"]}

try:
    city.check_params(typo)
    message = "(no error)"
except ValueError as err:
    message = str(err)

pd.Series(
    {
        "correct names": float(city.log_likelihood(good)),
        'misspelled "recovry" (lenient, used 0.25)': float(city.log_likelihood(typo)),
        "check_params on the misspelled dict": message,
    },
    name="log-likelihood or message",
).to_frame()

## A joint model

A joint model is a numpyro function. Three helpers keep each `BayesianModel` in its own namespace:

- `bm.sample_params("city.")` samples every prior of `bm` under a prefix, so the sites are `city.infection` and `city.recovery`, and returns a dict of **local** names (`infection`, `recovery`) that the model understands.
- `given={"recovery": value}` supplies a value instead of sampling it. The value is recorded as a deterministic site (`city.recovery`), so it appears in the posterior under the same name as a sampled one.
- `bm.factor("city", params)` adds the model's log-likelihood to the joint density. It is `numpyro.factor("city", bm.log_likelihood(params, strict=True))`.

Here the log of each model's recovery rate comes from `Normal(mu, tau)`: `mu` is the typical log rate and `tau` says how much the models differ. The hierarchy is sampled first and handed to each model as its `recovery`. The next section explains why the two recovery lines are written differently.

Wrapping the function in `NumpyroPosterior` lets the workflow stages use it as they use a `BayesianModel`. Its `prior_names()` shows the global site names: prefixed, with no bare `recovery`. The given values, `city.recovery` and `cohort.recovery`, are not sampled, so they are not listed, but they are recorded in the posterior.

In [ ]:
def joint_model() -> None:
    mu = numpyro.sample("mu", dist.Normal(np.log(0.25), 1.0))
    tau = numpyro.sample("tau", dist.HalfNormal(0.3))

    # City: non-centred.
    z_city = numpyro.sample("z_city", dist.Normal(0.0, 1.0))
    log_recovery_city = mu + tau * z_city
    # Cohort: centred.
    log_recovery_cohort = numpyro.sample("log_recovery_cohort", dist.Normal(mu, tau))

    p = city.sample_params("city.", given={"recovery": jnp.exp(log_recovery_city)})
    city.factor("city", p)
    p = cohort.sample_params("cohort.", given={"recovery": jnp.exp(log_recovery_cohort)})
    cohort.factor("cohort", p)


joint = NumpyroPosterior(joint_model)
pd.Series({"joint": joint.prior_names()}, name="prior_names()")

## Centred or non-centred?

A hierarchy can be written two ways. *Centred*: `log r ~ Normal(mu, tau)`. *Non-centred*: draw `z ~ Normal(0, 1)` and set `log r = mu + tau * z`. They describe the same prior, but NUTS sees different geometry. The choice depends on how much a member's own data say about its parameter.

- A member whose data barely inform it (the city: 20 days of an epidemic that has only started to grow) is written **non-centred**. Its posterior stays close to its prior, and the non-centred form removes the funnel between `tau` and the member value.
- A member whose data pin the parameter down (the cohort: 300 people followed until most have recovered) is written **centred**. The data fix `log r`, so `z` would have to shrink as `tau` shrinks, which creates the funnel.

The wrong choice shows up as divergent NUTS transitions at small `tau`, which the warmup check in the next section reports. This page does not run both versions. If a fit diverges, try changing the form of the member whose data are most informative first.

## Fitting the joint model

We use the staged workflow from the calibration notebooks, in small sizes:

1. `wf.lhs` draws a space-filling design of starting points, and `wf.evaluate` scores them;
2. `wf.optimize` runs a gradient optimiser from the best few;
3. NUTS starts its two chains from the optimiser's ends, and `wf.warmup_until` extends warmup until the chains agree;
4. `wf.sample_until` then draws the posterior samples.

`warm.converged` says whether warmup met its rules, which include a limit on divergent transitions. With only two members the spread `tau` is weakly determined, so we raise `target_accept_prob` to 0.99: NUTS takes smaller steps and avoids the divergences that the default 0.8 would leave.

In [ ]:
design = wf.evaluate(joint, wf.lhs(joint, 48, seed=0), batch_size=16)
opt = wf.optimize(
    joint,
    design.best(8),
    method=wf.Optax(optax.adam(0.05)),
    chunk_steps=50,
    max_steps=300,
    seed=0,
)


def make_mcmc(num_warmup: int) -> MCMC:
    return MCMC(
        NUTS(joint.numpyro_model(), target_accept_prob=0.99),
        num_warmup=num_warmup,
        num_samples=200,
        num_chains=2,
    )


warm = wf.warmup_until(
    make_mcmc,
    60,
    wf.WarmupRule(min_warmup=80, rhat=1.15),
    init_params=opt.candidates.init_params(2, seed=3),
    seed=3,
)
print(f"warmup steps: {warm.num_warmup}, converged: {warm.converged}")
run = wf.sample_until(warm.mcmc, wf.StopRule(rhat=None, ess=None, max_samples=200), seed=3)

The box plots show the posterior recovery rate of each model, with the true value as a gold star. The cohort's rate is pinned down by its own data. The city's is wider.

In [ ]:
fig = go.Figure()
names = ["city.recovery", "cohort.recovery"]
for name in names:
    fig.add_box(y=np.asarray(run.samples[name]).reshape(-1), name=name)
fig.add_scatter(
    x=names,
    y=[true[n] for n in names],
    mode="markers",
    marker={"symbol": "star", "size": 14, "color": "gold", "line": {"width": 1}},
    name="true value",
)
fig.update_layout(
    title="Posterior recovery rate in the joint fit",
    xaxis_title="model",
    yaxis_title="recovery rate (per day)",
)
fig.show()

## Replay each model from the joint posterior

The joint posterior holds sites for both models, plus `mu`, `tau` and the offsets, so it cannot be passed to a model's `posterior_runs` as it is. `bm.local_params(draws, prefix)` picks out the sites that belong to one model and returns them under the model's own names. Passing it as `params_fn` lets `posterior_runs` replay that model on every draw. The plot shows the median and the 10–90% band against the data.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["city", "cohort"])
members = {"city": (city, city_times, city_obs), "cohort": (cohort, cohort_times, cohort_obs)}
for col, (name, (bm, times, obs)) in enumerate(members.items(), start=1):
    runs = bm.posterior_runs(
        run.idata,
        n=100,
        params_fn=lambda d, bm=bm, name=name: bm.local_params(d, f"{name}."),
    )
    band = runs.quantiles((0.1, 0.5, 0.9))["baseline"]["I"]
    t = band.index.to_numpy()
    show = col == 1
    fig.add_scatter(
        x=t, y=band["0.9"], mode="lines", line={"width": 0}, showlegend=False, row=1, col=col
    )
    fig.add_scatter(
        x=t,
        y=band["0.1"],
        mode="lines",
        line={"width": 0},
        fill="tonexty",
        fillcolor="rgba(99,110,250,0.25)",
        name="10–90%",
        showlegend=show,
        row=1,
        col=col,
    )
    fig.add_scatter(
        x=t,
        y=band["0.5"],
        mode="lines",
        line={"color": "rgb(99,110,250)"},
        name="median",
        showlegend=show,
        row=1,
        col=col,
    )
    fig.add_scatter(
        x=times,
        y=obs,
        mode="markers",
        marker={"color": "black", "size": 6},
        name="data",
        showlegend=show,
        row=1,
        col=col,
    )
    fig.update_xaxes(title_text="day", row=1, col=col)
    fig.update_yaxes(title_text="infectious", row=1, col=col)
fig.update_layout(title="Each model replayed from the joint posterior", height=440)
fig.show()

## Things to watch

- **A given value replaces the model's prior, support included.** `Uniform("recovery", 0.02, 1.0)` is not applied to a value passed in `given=`. The model must run sensibly everywhere the hierarchy can reach, including starting points that the optimiser or NUTS may try.
- **`wf.lhs` stratifies only parentless scalar sites.** Plate offsets and centred members, which depend on other sites, are filled in by numpyro's `Predictive` instead.
- **Within one compiled model a name has one value.** Two values of "the same" parameter need two names, or a stratified parameter. A prefix separates models, not values inside one model.
- **Do not reuse a prefix.** Two models sampled with the same prefix fail with numpyro's duplicate-site error.
- **Read the warmup check.** If `warm.converged` is `False`, look at divergences and the centring choice before you trust the samples.

`examples/notebooks/38-calibration-multi-model.ipynb` is a three-model example (two cities and a cohort) that also compares the joint fit with fitting one city alone.

## Things to try

- Change the true cohort recovery rate from `0.40` to `0.28`, the city's value. What happens to the posterior for `tau`?
- Write the cohort non-centred and the city centred, and compare `warm.converged` and `warm.num_warmup`.
- Misspell a key in `given=`, such as `"recovry"`, and read the error from `sample_params`.